# 🐌 Single-GPU Baseline — BERT Fine-tuning on IMDb
### TP Deep Learning Distribué — Expérience A : entraînement sur 1 seul GPU

---
**Objectif :** établir la référence (baseline) en entraînant BERT pour la classification de sentiments (IMDb) sur **un seul GPU**, mesurer le temps d'entraînement, et montrer les limites mémoire (OOM) atteintes quand on augmente la taille de batch sans parallélisme.

Ce notebook produit un fichier `results_single_gpu.json` qui sera réutilisé par le notebook `multi-gpu-experiment.ipynb` pour calculer le **speedup réel**.

**Environnement requis :** Kaggle Notebook avec accélérateur **GPU T4 x2** activé (on n'utilise volontairement qu'un seul des deux GPU dans ce notebook).

---

## ⚙️ SECTION 1 — Environnement
Installe les dépendances et vérifie le matériel disponible.

In [1]:
# ───────────────────────────────────────────
# STEP 1 — Install all required libraries
# ───────────────────────────────────────────
!pip install -q "transformers>=4.41.0"
!pip install -q "datasets>=2.20.0"
!pip install -q "pytorch-lightning==2.2.4"
!pip install -q "torchmetrics==1.4.0"
print("✅ All libraries installed successfully")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 802.2/802.2 kB 14.1 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 868.8/868.8 kB 15.4 MB/s eta 0:00:00a 0:00:01
✅ All libraries installed successfully


In [2]:
# ───────────────────────────────────────────
# STEP 2 — Verify GPU setup
# ───────────────────────────────────────────
import torch

num_gpus = torch.cuda.device_count()
print(f"Number of GPUs available: {num_gpus}")

for i in range(num_gpus):
    name = torch.cuda.get_device_name(i)
    mem  = torch.cuda.get_device_properties(i).total_memory / 1e9
    print(f"  GPU {i}: {name} — {mem:.1f} GB VRAM")

if num_gpus == 0:
    print("\n⚠️  No GPU detected. Enable an accelerator in Kaggle Settings.")
else:
    print(f"\n✅ {num_gpus} GPU(s) detected. This notebook only needs ONE GPU (device 0).")

DEVICE_ID = 0
torch.cuda.set_device(DEVICE_ID)
print(f"Using cuda:{DEVICE_ID} for this single-GPU baseline.")

Number of GPUs available: 2
  GPU 0: Tesla T4 — 15.6 GB VRAM
  GPU 1: Tesla T4 — 15.6 GB VRAM

✅ 2 GPU(s) detected. This notebook only needs ONE GPU (device 0).
Using cuda:0 for this single-GPU baseline.


## 📦 SECTION 2 — Load and Prepare the IMDb Dataset

In [3]:
# ───────────────────────────────────────────
# STEP 3 — Download IMDb dataset from HuggingFace
# Size: ~80MB — downloads automatically, no Kaggle account needed
# ───────────────────────────────────────────
from datasets import load_dataset

print("Downloading IMDb dataset...")
raw_dataset = load_dataset("imdb")

print("\n✅ Dataset loaded!")
print(f"   Train samples : {len(raw_dataset['train'])}")
print(f"   Test  samples : {len(raw_dataset['test'])}")
print(f"   Classes       : 0 = Negative, 1 = Positive")

sample = raw_dataset['train'][0]
print(f"\n--- Sample Review (first 200 chars) ---")
print(sample['text'][:200], "...")
print(f"Label: {'POSITIVE ✅' if sample['label'] == 1 else 'NEGATIVE ❌'}")

README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]


✅ Dataset loaded!
   Train samples : 25000
   Test  samples : 25000
   Classes       : 0 = Negative, 1 = Positive

--- Sample Review (first 200 chars) ---
I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ev ...
Label: NEGATIVE ❌


In [4]:
# ───────────────────────────────────────────
# STEP 4 — Tokenize the dataset with BertTokenizer
# What happens: raw text → token IDs that BERT understands
# ───────────────────────────────────────────
from transformers import BertTokenizer

print("Loading BERT tokenizer...")
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        padding="max_length",   # pad short reviews to 128 tokens
        truncation=True,        # cut long reviews at 128 tokens
        max_length=128          # 128 keeps VRAM usage manageable
    )

print("Tokenizing dataset (this takes ~2 minutes)...")
tokenized = raw_dataset.map(tokenize_function, batched=True)
tokenized.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

print("\n✅ Tokenization complete!")
print("--- Before tokenization ---")
print(f"Raw text: '{raw_dataset['train'][0]['text'][:60]}...'")
print("\n--- After tokenization ---")
print(f"input_ids shape : {tokenized['train'][0]['input_ids'].shape}")
print(f"First 10 tokens : {tokenized['train'][0]['input_ids'][:10]}")
print(f"Decoded back     : {tokenizer.decode(tokenized['train'][0]['input_ids'][:10])}")

Loading BERT tokenizer...


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenizing dataset (this takes ~2 minutes)...


Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]


✅ Tokenization complete!
--- Before tokenization ---
Raw text: 'I rented I AM CURIOUS-YELLOW from my video store because of ...'

--- After tokenization ---
input_ids shape : torch.Size([128])
First 10 tokens : tensor([  101,  1045, 12524,  1045,  2572,  8025,  1011,  3756,  2013,  2026])
Decoded back     : [CLS] i rented i am curious - yellow from my


In [5]:
# ───────────────────────────────────────────
# STEP 5 — Create PyTorch DataLoaders
# On single GPU there is no sharding: one process sees the whole subset.
# ───────────────────────────────────────────
from torch.utils.data import DataLoader, random_split

# Dataset volontairement large : le calcul GPU doit dominer les coûts fixes
# (téléchargement, tokenisation, démarrage torchrun) pour que le speedup soit visible.
TRAIN_SIZE = 20000   # sur les 25 000 dispos dans IMDb train
VAL_SIZE   = 5000    # sur les 25 000 dispos dans IMDb test
BATCH_SIZE = 32   # single GPU baseline batch size

g = torch.Generator().manual_seed(42)   # fixed seed so both notebooks see the SAME split
train_subset, _ = random_split(tokenized['train'], [TRAIN_SIZE, len(tokenized['train']) - TRAIN_SIZE], generator=g)
val_subset,   _ = random_split(tokenized['test'],  [VAL_SIZE,   len(tokenized['test'])  - VAL_SIZE], generator=g)

train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_subset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"✅ DataLoaders ready")
print(f"   Train batches : {len(train_loader)} batches × {BATCH_SIZE} samples")
print(f"   Val   batches : {len(val_loader)} batches × {BATCH_SIZE} samples")

✅ DataLoaders ready
   Train batches : 625 batches × 32 samples
   Val   batches : 157 batches × 32 samples


## 🧠 SECTION 3 — Define the BERT Model with PyTorch Lightning

In [6]:
# ───────────────────────────────────────────
# STEP 6 — Define the Lightning Model
# LightningModule wraps BERT and handles train/val logic.
# This EXACT class is reused unchanged in train_ddp.py for Experiment B,
# so the only variable between experiments is the parallelization strategy.
# ───────────────────────────────────────────
import pytorch_lightning as pl
import torch.nn as nn
from transformers import BertForSequenceClassification
from torch.optim import AdamW
from torchmetrics.classification import BinaryAccuracy

class BERTSentimentClassifier(pl.LightningModule):
    """
    BERT fine-tuned for binary sentiment classification.
    Architecture:
        Input text → BertTokenizer → BERT (110M params) → [CLS] vector → Linear(768→2) → Softmax
    """

    def __init__(self, learning_rate=2e-5):
        super().__init__()
        self.save_hyperparameters()

        self.model = BertForSequenceClassification.from_pretrained(
            "bert-base-uncased",
            num_labels=2
        )

        self.train_acc = BinaryAccuracy()
        self.val_acc   = BinaryAccuracy()

    def forward(self, input_ids, attention_mask, labels=None):
        return self.model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

    def training_step(self, batch, batch_idx):
        """
        One training step = one mini-batch.
        Under DDP each GPU runs this INDEPENDENTLY on its own shard of data,
        then gradients are synchronized across GPUs via AllReduce (NCCL)
        before the optimizer step
        """
        outputs = self(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
            labels=batch["label"]
        )
        loss   = outputs.loss
        logits = outputs.logits
        preds  = torch.argmax(logits, dim=1)

        self.train_acc(preds, batch["label"])
        self.log("train_loss", loss,           prog_bar=True, sync_dist=True)
        self.log("train_acc",  self.train_acc, prog_bar=True, sync_dist=True)
        return loss

    def validation_step(self, batch, batch_idx):
        """sync_dist=True averages metrics across all GPUs (the reduction step)."""
        outputs = self(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
            labels=batch["label"]
        )
        loss   = outputs.loss
        logits = outputs.logits
        preds  = torch.argmax(logits, dim=1)

        self.val_acc(preds, batch["label"])
        self.log("val_loss", loss,         prog_bar=True, sync_dist=True)
        self.log("val_acc",  self.val_acc, prog_bar=True, sync_dist=True)

    def configure_optimizers(self):
        return AdamW(self.parameters(), lr=self.hparams.learning_rate, weight_decay=0.01)


print("✅ BERTSentimentClassifier defined")
print("   Parameters: ~110 million (bert-base-uncased)")
print("   Input : input_ids + attention_mask (shape: [batch, 128])")
print("   Output: 2 logits → Positive or Negative")

/usr/local/lib/python3.12/dist-packages/lightning_fabric/__init__.py:41: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.


✅ BERTSentimentClassifier defined
   Parameters: ~110 million (bert-base-uncased)
   Input : input_ids + attention_mask (shape: [batch, 128])
   Output: 2 logits → Positive or Negative


## 🐌 SECTION 4 — Experiment A: Single-GPU Training (the baseline)

In [7]:
# ───────────────────────────────────────────
# STEP 7 — Train on 1 GPU
# Record the time — this is the reference against which DDP is compared
# ───────────────────────────────────────────
import time

# Avoids a matmul-precision warning that can bury the timing prints below
torch.set_float32_matmul_precision('medium')


class EpochTimer(pl.Callback):
    """Prints the wall-clock time of each epoch AS IT HAPPENS, independently
    of the progress bar. This is what lets you actually see the training
    move (instead of waiting silently until the very end)."""
    def on_train_epoch_start(self, trainer, pl_module):
        self._t0 = time.time()

    def on_train_epoch_end(self, trainer, pl_module):
        dt = time.time() - self._t0
        print(f"  ⏱️  Epoch {trainer.current_epoch + 1}/{trainer.max_epochs} "
              f"terminée en {dt:.1f}s", flush=True)


print("=" * 55)
print("EXPERIMENT A: Single GPU Training")
print("=" * 55)

model_single = BERTSentimentClassifier(learning_rate=2e-5)

trainer_single = pl.Trainer(
    accelerator="gpu",
    devices=1,              # ← Only 1 GPU
    strategy="auto",        # ← No distribution
    max_epochs=2,
    precision="16-mixed",   # Mixed precision to fit in VRAM
    enable_progress_bar=False,   # ← the spinning bar hides the epoch prints; turn it off
    callbacks=[EpochTimer()],    # ← explicit, flushed, always-visible timing
    log_every_n_steps=50,
    enable_checkpointing=False,
)

print(f"\n📋 Config: 1 GPU | Batch={BATCH_SIZE} | Train={TRAIN_SIZE} | Val={VAL_SIZE} | Epochs=2")
print("Starting training...\n", flush=True)

start_single = time.time()
trainer_single.fit(model_single, train_loader, val_loader)
time_single = time.time() - start_single

results_single = trainer_single.callback_metrics

print("\n" + "=" * 55)
print("EXPERIMENT A — RESULTS")
print("=" * 55)
print(f"  Total training time : {time_single:.1f} seconds ({time_single/60:.1f} minutes)")
print(f"  Final train loss    : {results_single.get('train_loss', 'N/A')}")
print(f"  Final val accuracy  : {results_single.get('val_acc', 'N/A')}")


EXPERIMENT A: Single GPU Training


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Using 16bit Automatic Mixed Precision


📋 Config: 1 GPU | Batch=32 | Train=20000 | Val=5000 | Epochs=2
Starting training...



Missing logger folder: /kaggle/working/lightning_logs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]

  | Name      | Type                          | Params
------------------------------------------------------------
0 | model     | BertForSequenceClassification | 109 M 
1 | train_acc | BinaryAccuracy                | 0     
2 | val_acc   | BinaryAccuracy                | 0     
------------------------------------------------------------
109 M     Trainable params
0         Non-trainable params
109 M     Total params
437.935   Total estimated model params size (MB)
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


  ⏱️  Epoch 1/2 terminée en 160.3s
  ⏱️  Epoch 2/2 terminée en 164.3s


`Trainer.fit` stopped: `max_epochs=2` reached.



EXPERIMENT A — RESULTS
  Total training time : 340.5 seconds (5.7 minutes)
  Final train loss    : 0.07906454801559448
  Final val accuracy  : 0.8722000122070312


## 💥 Experiment A2: Out-of-Memory demo
**Expected result:** increasing the batch size to 256 on a single GPU should crash with a CUDA OOM error. This is exactly the problem DDP solves in the `multi-gpu-experiment.ipynb` notebook (splitting 256 into 128 per GPU).

In [8]:
# ───────────────────────────────────────────
# STEP 8 — Reproduce the Out-of-Memory error
# This proves WHY we need DDP
#
# NOTE: batch_size=256 does NOT reliably OOM a T4 for BERT-base at seq_len=128
# in fp16 — the activations/gradients/optimizer states just barely fit in
# 16GB. 1024 pushes comfortably past that ceiling regardless of the exact
# GPU (T4/P100/etc.), so the demo triggers a real OOM every time.
# ───────────────────────────────────────────
OOM_BATCH_SIZE = 1024   

print("=" * 55)
print("EXPERIMENT A2: Single GPU — Large Batch (OOM Demo)")
print("=" * 55)
print(f"Trying batch_size={OOM_BATCH_SIZE} on 1 GPU...\n")

large_batch_loader = DataLoader(
    train_subset,
    batch_size=OOM_BATCH_SIZE,   # ← This will crash on 1 GPU
    shuffle=True,
    num_workers=2
)

model_oom = BERTSentimentClassifier()

trainer_oom = pl.Trainer(
    accelerator="gpu",
    devices=1,
    max_epochs=1,
    precision="16-mixed",
    enable_checkpointing=False,
)

oom_confirmed = False
try:
    trainer_oom.fit(model_oom, large_batch_loader, val_loader)
    print("\n(No OOM error — your GPU might have more memory than expected. "
          f"Try raising OOM_BATCH_SIZE above {OOM_BATCH_SIZE} and re-running this cell.)")
except RuntimeError as e:
    if "out of memory" in str(e).lower():
        oom_confirmed = True
        print("\n" + "❌ " * 15)
        print("CUDA OUT OF MEMORY ERROR — as expected!")
        print(f"This is exactly why we need DDP: split {OOM_BATCH_SIZE} into "
              f"{OOM_BATCH_SIZE // 2}+{OOM_BATCH_SIZE // 2} across 2 GPUs "
              "(or use gradient accumulation on 1 GPU).")
        print(f"Error: {str(e)[:200]}")
        print("❌ " * 15)
    else:
        raise e

del model_oom, trainer_oom
torch.cuda.empty_cache()

EXPERIMENT A2: Single GPU — Large Batch (OOM Demo)
Trying batch_size=1024 on 1 GPU...



Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Using 16bit Automatic Mixed Precision

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/loops/fit_loop.py:298: The number of training batches (20) is smaller than the logging interval Trainer(log_every_n_steps=50). Set a lower value for log_every_n_steps if you want to see logs for the training epoch.


Training: |          | 0/? [00:00<?, ?it/s]


❌ ❌ ❌ ❌ ❌ ❌ ❌ ❌ ❌ ❌ ❌ ❌ ❌ ❌ ❌ 
CUDA OUT OF MEMORY ERROR — as expected!
This is exactly why we need DDP: split 1024 into 512+512 across 2 GPUs (or use gradient accumulation on 1 GPU).
Error: CUDA out of memory. Tried to allocate 192.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 70.81 MiB is free. Including non-PyTorch memory, this process has 14.49 GiB memory in use. Of the all
❌ ❌ ❌ ❌ ❌ ❌ ❌ ❌ ❌ ❌ ❌ ❌ ❌ ❌ ❌ 


## 💾 SECTION 5 — Save Results for Cross-Notebook Comparison

In [9]:
# ───────────────────────────────────────────
# STEP 9 — Save the baseline results to disk
# multi-gpu-experiment.ipynb reads this file to compute the real speedup.
# ───────────────────────────────────────────
import json, os

OUTPUT_DIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."

summary = {
    "experiment"   : "A - single GPU baseline",
    "num_gpus"     : 1,
    "batch_size"   : BATCH_SIZE,
    "epochs"       : 2,
    "train_size"   : TRAIN_SIZE,
    "val_size"     : VAL_SIZE,
    "time_seconds" : time_single,
    "train_loss"   : float(results_single.get("train_loss", float("nan"))),
    "val_acc"      : float(results_single.get("val_acc", float("nan"))),
    "oom_at_batch256_on_1_gpu": oom_confirmed,
}

with open(os.path.join(OUTPUT_DIR, "results_single_gpu.json"), "w") as f:
    json.dump(summary, f, indent=2)

print("✅ Saved results_single_gpu.json:")
print(json.dumps(summary, indent=2))

✅ Saved results_single_gpu.json:
{
  "experiment": "A - single GPU baseline",
  "num_gpus": 1,
  "batch_size": 32,
  "epochs": 2,
  "train_size": 20000,
  "val_size": 5000,
  "time_seconds": 340.4922354221344,
  "train_loss": 0.07906454801559448,
  "val_acc": 0.8722000122070312,
  "oom_at_batch256_on_1_gpu": true
}


## 🔍 SECTION 6 — Inference Demo
Test the single-GPU-trained model with your own custom reviews.

In [10]:
# ───────────────────────────────────────────
# STEP 10 — Predict sentiment on custom reviews
# Change the reviews below to test with your own text!
# ───────────────────────────────────────────
import torch.nn.functional as F

def predict_sentiment(text, model, tokenizer, device="cuda"):
    """Run inference on a single text string."""
    model.eval()
    model = model.to(device)

    encoding = tokenizer(
        text,
        return_tensors="pt",
        padding="max_length",
        truncation=True,
        max_length=128
    )

    input_ids      = encoding["input_ids"].to(device)
    attention_mask = encoding["attention_mask"].to(device)

    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        probs   = F.softmax(outputs.logits, dim=1)
        pred    = torch.argmax(probs, dim=1).item()
        conf    = probs[0][pred].item()

    label = "POSITIVE ✅" if pred == 1 else "NEGATIVE ❌"
    return label, conf


custom_reviews = [
    "This movie was absolutely incredible. The acting was phenomenal and I was on the edge of my seat the entire time.",
    "Terrible film. Boring plot, bad acting, and a complete waste of 2 hours. I almost fell asleep.",
    "It was okay. Some parts were good but overall the story felt a bit weak and predictable."
]

print("=" * 60)
print("INFERENCE DEMO — Sentiment Predictions (single-GPU model)")
print("=" * 60)

inference_model = model_single.to("cuda:0")

for i, review in enumerate(custom_reviews, 1):
    label, confidence = predict_sentiment(review, inference_model, tokenizer, "cuda:0")
    print(f"\nReview {i}: \"{review[:70]}...\"")
    print(f"  Prediction  : {label}")
    print(f"  Confidence  : {confidence*100:.1f}%")

print("\n" + "=" * 60)

INFERENCE DEMO — Sentiment Predictions (single-GPU model)

Review 1: "This movie was absolutely incredible. The acting was phenomenal and I ..."
  Prediction  : POSITIVE ✅
  Confidence  : 98.8%

Review 2: "Terrible film. Boring plot, bad acting, and a complete waste of 2 hour..."
  Prediction  : NEGATIVE ❌
  Confidence  : 99.5%

Review 3: "It was okay. Some parts were good but overall the story felt a bit wea..."
  Prediction  : NEGATIVE ❌
  Confidence  : 95.6%

